# Asymptote Fits for Contactome Graph Invariants

Load the precomputed contactome invariant measurements across checkpoints and fit finite-size correction curves to estimate their limiting behavior.

In [ ]:

import json
import pathlib

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display
from refig import refig

from zfish.asymptote import (
    PROGRESS_LEVELS,
    PLOT_EXCLUDE,
    DEFAULT_MODEL_SPECS,
    average_milestones,
    build_summary_lines,
    chunked,
    compute_checkpoint_metadata,
    estimate_progress_edit,
    fit_invariants,
    get_limit_value,
    predict_from_fit,
)

plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["font.family"] = "Roboto"
plt.rcParams["figure.dpi"] = 150

DATA_PATH = pathlib.Path("contactome_invariants_per_graph.json")
if not DATA_PATH.exists():
    DATA_PATH = pathlib.Path("../notebooks/contactome_invariants_per_graph.json")
if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Could not locate contactome_invariants_per_graph.json relative to this notebook."
    )

FIGURE_DIR = pathlib.Path("figures/contactomes")
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:

with open(DATA_PATH) as f:
    invariants_per_graph = json.load(f)

checkpoint_labels, checkpoint_edits, checkpoint_lookup, DEFAULT_SHIFT = compute_checkpoint_metadata(
    invariants_per_graph
)

print(
    f"Loaded {len(invariants_per_graph)} invariants across "
    f"{len(checkpoint_labels)} checkpoints."
)
print(f"Checkpoint shift (delta) for feature transforms: {DEFAULT_SHIFT:.0f} edits")


In [ ]:

MODEL_SPECS = DEFAULT_MODEL_SPECS

fit_results, skipped = fit_invariants(
    invariants_per_graph,
    checkpoint_labels,
    checkpoint_lookup,
    MODEL_SPECS,
    shift=DEFAULT_SHIFT,
)

print(f"Fitted asymptotes for {len(fit_results)} invariants.")
if skipped:
    print(
        "Skipped due to insufficient data or monotonic fits:",
        ", ".join(skipped),
    )


In [ ]:

summary_lines = build_summary_lines(fit_results, progress_levels=PROGRESS_LEVELS)

display(Markdown("\n".join(summary_lines)))

# Generate an SVG figure of the table:
fig = plt.figure(figsize=(12, len(summary_lines) * 0.4))
ax = fig.add_subplot(1, 1, 1)
ax.axis("off")
table_data = [line.strip().split("|")[1:-1] for line in summary_lines]
table = ax.table(
    cellText=table_data[1:],
    colLabels=table_data[0],
    cellLoc="center",
    loc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)
fig.tight_layout()
figure_path = FIGURE_DIR / "invariant_asymptote_summary.svg"
refig.savefig(str(figure_path))


In [ ]:

with plt.style.context("seaborn-v0_8-whitegrid"):
    sorted_results = sorted(fit_results, key=lambda item: item["invariant"])
    plot_results = [
        result
        for result in sorted_results
        if result["invariant"] not in PLOT_EXCLUDE
    ]
    group_size = 10
    future_limit = 100_000.0
    tick_min = 0
    tick_max = 100_000.0
    tick_inc = 10000.0
    xtick_positions = np.arange(tick_min, tick_max + 1e-9, tick_inc)
    xtick_labels = [f"{int(pos/1000)}K" for pos in xtick_positions]
    milestone_color = "#6a3d9a"
    for group_idx, group in enumerate(chunked(plot_results, group_size), start=1):
        fig, axes = plt.subplots(2, 5, figsize=(10, 3), sharex=True)
        axes = axes.flatten()
        for ax, result in zip(axes, group):
            x = result["x"]
            y = result["y"]
            best = result["best_fit"]
            limit_value = get_limit_value(best)
            last_observed = x.max()
            x_grid = np.linspace(x.min(), max(future_limit, last_observed), 600)
            y_grid = predict_from_fit(x_grid, best)

            ax.scatter(x, y, color="#2c7fb8", s=18)

            past_mask = x_grid <= last_observed
            ax.plot(
                x_grid[past_mask],
                y_grid[past_mask],
                color="#f03b20",
            )
            if np.any(~past_mask):
                ax.plot(
                    x_grid[~past_mask],
                    y_grid[~past_mask],
                    color="#f03b20",
                    alpha=0.4,
                    linestyle="--",
                )

            if limit_value is not None:
                ax.axhline(
                    limit_value,
                    color="#555555",
                    linestyle="--",
                    linewidth=1,
                )

            milestone_values = []
            for level in PROGRESS_LEVELS:
                edit_value = estimate_progress_edit(result, level, limit_value)
                if edit_value is None or edit_value > future_limit:
                    continue
                milestone_values.append((level, edit_value))

            if future_limit > last_observed:
                ax.axvspan(
                    last_observed,
                    future_limit,
                    color="#f03b20",
                    alpha=0.05,
                )

            ax.set_title(result["invariant"])
            ax.set_xlim(0, future_limit)
            ax.set_xticks(xtick_positions)
            ax.set_xticklabels(xtick_labels, rotation=90)
            ax.set_yticklabels(
                [f"{ytick:.2g}" for ytick in ax.get_yticks()],
                rotation=45,
            )

            ymin, ymax = ax.get_ylim()
            for level, edit_value in milestone_values:
                ax.axvline(
                    edit_value,
                    color=milestone_color,
                    linestyle=":",
                    linewidth=1.5,
                    alpha=0.65,
                )
                ax.annotate(
                    f"{int(level * 100)}%",
                    xy=(edit_value, ymax),
                    xytext=(0, -10),
                    textcoords="offset points",
                    rotation=90,
                    ha="center",
                    va="top",
                    fontsize=7,
                    color=milestone_color,
                    alpha=0.8,
                )

        for ax in axes:
            ax.grid(True, linestyle=":", linewidth=0.5)

        for ax in axes[len(group):]:
            ax.axis("off")

        fig.tight_layout(rect=(0, 0, 1, 1))

        output_path = FIGURE_DIR / f"asymptote-grid-{group_idx:02d}.svg"
        refig.savefig(str(output_path))
        plt.show()


In [ ]:

average_milestones_mean, average_milestones_std, average_milestones_formatted = average_milestones(
    fit_results, progress_levels=PROGRESS_LEVELS
)
average_milestones_formatted
